# 05.4 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/04-xgboost-lightgbm/exercises.ipynb)

Answers in [solutions.ipynb](solutions.ipynb).

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

In [ ]:
import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]
cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

def cv_auc(model):
    """Mean 5-fold cross-validated AUC on the training customers."""
    return cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc").mean()

def test_auc(model):
    """Fit on all training customers, score AUC on the held-out test customers."""
    model.fit(X_train, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print(f"Benchmark logistic regression: CV AUC {cv_auc(logreg):.3f}, test AUC {test_auc(logreg):.3f}")

import time
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier

def make_models():
    return {
        "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=42, n_jobs=4),
        "XGBoost": xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8,
                                     colsample_bytree=0.8, random_state=42, n_jobs=4),
        "LightGBM": lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=8, min_child_samples=30,
                                       subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                       random_state=42, n_jobs=4, verbose=-1),
    }

## Exercise 1: early stopping for LightGBM

Repeat the early-stopping experiment with LightGBM: `n_estimators=2000`, `learning_rate=0.05`, and `callbacks=[lgb.early_stopping(50, verbose=False)]` in `fit`, with `eval_set=[(X_val, y_val)]` and `eval_metric="auc"`. At which round does it stop? What is the test AUC?

In [ ]:
# your code here

## Exercise 2: depth

For XGBoost with 300 trees and learning rate 0.05, compare `max_depth` = 1, 2, 3, 6, 10 by CV AUC. Depth 1 means stumps: each tree uses one feature, so the model cannot learn interactions at all. What does the result tell you about interactions in the churn data?

In [ ]:
# your code here

## Exercise 3: depth on fraud

Do the same depth comparison on the fraud orders. Fraud is defined with "and", which a single stump cannot express. Do you expect stumps to fall behind? Check.

In [ ]:
def make_orders(n, seed=0):
    """Orders like lesson 00.1: fraud = big order from a new account, OR very big order to a mismatched country."""
    rng = np.random.default_rng(seed)
    total = rng.gamma(2.0, 150, n)
    account_days = rng.integers(1, 1000, n)
    account_days = np.where(rng.random(n) < 0.25, rng.integers(1, 15, n), account_days)
    items = rng.integers(1, 15, n)
    country_match = rng.choice([0, 1], n, p=[0.15, 0.85])
    truth = ((total > 300) & (account_days < 10)) | ((country_match == 0) & (total > 500))
    fraud = (truth ^ (rng.random(n) < 0.03)).astype(int)        # 3% label noise
    return pd.DataFrame({"total": total, "account_days": account_days, "items": items,
                         "country_match": country_match, "fraud": fraud})

orders = make_orders(20_000)
fX, fy = orders.drop(columns="fraud"), orders["fraud"]
fX_train, fX_test, fy_train, fy_test = train_test_split(fX, fy, test_size=0.25, random_state=42, stratify=fy)

In [ ]:
# your code here

## Exercise 4: how much data does boosting need?

Train logistic regression and XGBoost on the first 500, 2,000, 5,000 and 15,000 fraud training orders, always scoring on the same test set. Plot test AUC against training size for both.

In [ ]:
# your code here